# WM Prediction 2026 – Version 2 (mit Elo Ratings)

Diese Version erweitert das Basismodell um dynamische Elo-Ratings.

**Architektur:**
- **Elo-Berechnung**: Gesamter Datensatz (1872–heute) – liefert historisch korrekte, gewachsene Ratings
- **Poisson-Modell**: Nur die letzten 20 Jahre – trainiert auf modernem Fußball

Vorteile:
- Gegnerstärke wird berücksichtigt
- Form wird indirekt abgebildet
- Besonders stark bei Nationalmannschaften
- Gute Basis für spätere Wettquoten-Integration

## 1 · Setup

In [12]:
%pip install statsmodels

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson

Note: you may need to restart the kernel to use updated packages.Defaulting to user installation because normal site-packages is not writeable




[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 · Daten laden

Wir laden den kompletten Datensatz einmal und leiten zwei Teilmengen ab:
- `all_matches` – alle Spiele seit 1872, **nur für die Elo-Berechnung**
- `model_matches` – letzte 20 Jahre, **nur für das Poisson-Modell**

In [13]:
RESULTS_FILE = "results.csv"

all_matches = pd.read_csv(RESULTS_FILE)
all_matches['date'] = pd.to_datetime(all_matches['date'])
all_matches = all_matches.sort_values('date').reset_index(drop=True)

cutoff = pd.Timestamp.now() - pd.DateOffset(years=20)
model_matches = all_matches[all_matches['date'] >= cutoff].copy()

print(f"Gesamt-Datensatz  (Elo):     {len(all_matches):>6} Spiele  "
      f"({all_matches['date'].dt.year.min()}–{all_matches['date'].dt.year.max()})")
print(f"Modell-Datensatz  (Poisson): {len(model_matches):>6} Spiele  "
      f"({model_matches['date'].dt.year.min()}–{model_matches['date'].dt.year.max()})")

Gesamt-Datensatz  (Elo):      49378 Spiele  (1872–2026)
Modell-Datensatz  (Poisson):  19345 Spiele  (2006–2026)


## 3 · Elo-Berechnung (gesamter Datensatz)

Die Elo-Ratings werden über **alle Spiele seit 1872** berechnet, damit die aktuellen
Ratings historisch korrekt gewachsen sind.  
Der Elo-Wert *vor* jedem Spiel wird in `all_matches` gespeichert und später
den `model_matches` zugeführt (als Feature für das Poisson-Modell).

In [14]:
# K-Faktor je Turnierkategorie
def get_k_factor(tournament):
    t = str(tournament)
    if t == "FIFA World Cup":            return 60
    if "UEFA Euro" in t:                 return 50
    if "Copa América" in t:              return 50
    if "African Cup of Nations" in t:    return 50
    if "AFC Asian Cup" in t:             return 50
    if "OFC Nations Cup" in t:           return 50
    if "Nations League" in t:            return 40
    if "qualification" in t.lower():     return 30
    if "Friendly" in t:                  return 15
    return 25

In [15]:
INITIAL_ELO = 1500
elo = {}

def get_elo(team):
    return elo.get(team, INITIAL_ELO)


elo_home_before = []
elo_away_before = []
current_year = None

for _, row in all_matches.iterrows():

    year = row['date'].year

    # Elo-Decay: 2 % Rückkehr Richtung Mittelwert pro Jahr
    if current_year is None:
        current_year = year
    if year > current_year:
        for team in elo:
            elo[team] = INITIAL_ELO + (elo[team] - INITIAL_ELO) * 0.98
        current_year = year

    home = row['home_team']
    away = row['away_team']
    eh   = get_elo(home)
    ea   = get_elo(away)

    elo_home_before.append(eh)
    elo_away_before.append(ea)

    expected_home = 1 / (1 + 10 ** ((ea - eh) / 400))

    if   row['home_score'] > row['away_score']: actual_home = 1.0
    elif row['home_score'] < row['away_score']: actual_home = 0.0
    else:                                       actual_home = 0.5

    k = get_k_factor(row['tournament'])
    elo[home] = eh + k * (actual_home       - expected_home)
    elo[away] = ea + k * ((1.0 - actual_home) - (1.0 - expected_home))


# Elo-Werte (vor dem jeweiligen Spiel) in all_matches schreiben
all_matches['elo_home'] = elo_home_before
all_matches['elo_away'] = elo_away_before
all_matches['elo_diff'] = all_matches['elo_home'] - all_matches['elo_away']

print(f"Elo-Berechnung abgeschlossen – {len(elo)} Teams im Elo-Verzeichnis.")

Elo-Berechnung abgeschlossen – 336 Teams im Elo-Verzeichnis.


## 4 · Aktuelle Elo-Tabelle

In [16]:
elo_table = (
    pd.DataFrame(list(elo.items()), columns=['team', 'elo'])
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
elo_table.index += 1   # Rang ab 1
elo_table.head(25)

,team,elo
1,Spain,1930.512426
2,Argentina,1902.740386
3,France,1880.847412
4,Morocco,1846.111521
5,England,1833.945515
6,Portugal,1831.771007
7,Brazil,1815.871151
8,Colombia,1814.160921
9,Japan,1808.122817
10,Germany,1805.952236


## 5 · Poisson-Modell (letzte 20 Jahre)

Die Elo-Features aus Schritt 3 werden den `model_matches` zugeführt.
Das Modell trainiert **ausschließlich auf diesen modernen Spielen**.

In [17]:
# Elo-Spalten aus all_matches in model_matches übernehmen
model_matches = model_matches.join(
    all_matches[['elo_home', 'elo_away', 'elo_diff']],
    how='left'
)

### 5.1 · Turnier- und Zeitgewichtung

In [18]:
def tournament_weight(tournament):
    t = str(tournament)
    if 'FIFA World Cup' in t:           return 20
    if 'UEFA Euro' in t:                return 15
    if 'Copa América' in t:             return 15
    if 'African Cup of Nations' in t:   return 15
    if 'AFC Asian Cup' in t:            return 15
    if 'OFC Nations Cup' in t:          return 15
    if 'Nations League' in t:           return 8
    if 'qualification' in t.lower():    return 8
    if 'Friendly' in t:                 return 1
    return 2

model_matches['tournament_weight'] = model_matches['tournament'].apply(tournament_weight)

days_old = (pd.Timestamp.now() - model_matches['date']).dt.days
model_matches['time_weight'] = np.exp(-days_old / 1460)
model_matches['weight'] = model_matches['tournament_weight'] * model_matches['time_weight']

### 5.2 · Modell-Datensatz aufbauen

In [19]:
home_df = model_matches[['home_team', 'away_team', 'home_score', 'weight', 'elo_diff']].copy()
home_df.columns = ['team', 'opponent', 'goals', 'weight', 'elo_diff']

away_df = model_matches[['away_team', 'home_team', 'away_score', 'weight', 'elo_diff']].copy()
away_df.columns = ['team', 'opponent', 'goals', 'weight', 'elo_diff']
away_df['elo_diff'] = -away_df['elo_diff']   # Perspektive umkehren

goal_model_data = pd.concat([home_df, away_df], ignore_index=True)
print(f"Trainingszeilen: {len(goal_model_data):,}  "
      f"(= {len(goal_model_data) // 2:,} Spiele x 2 Perspektiven)")

Trainingszeilen: 38,690  (= 19,345 Spiele x 2 Perspektiven)


### 5.3 · Modell trainieren

In [20]:
model = smf.glm(
    formula='goals ~ team + opponent + elo_diff',
    data=goal_model_data,
    family=sm.families.Poisson(),
    freq_weights=goal_model_data['weight']
).fit()

print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                38546
Model:                            GLM   Df Residuals:                 76017.37
Model Family:                 Poisson   Df Model:                          632
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0371e+05
Date:                Fri, 05 Jun 2026   Deviance:                       80547.
Time:                        13:21:35   Pearson chi2:                 7.26e+04
No. Iterations:                    21   Pseudo R-squ. (CS):             0.6964
Covariance Type:            nonrobust                                         
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------

## 6 · Vorhersagefunktionen

In [21]:
def predict_goals(team, opponent):
    """Erwartete Tore von `team` gegen `opponent` (aktueller Elo-Stand)."""
    elo_diff = get_elo(team) - get_elo(opponent)
    df = pd.DataFrame({'team': [team], 'opponent': [opponent], 'elo_diff': [elo_diff]})
    return float(model.predict(df)[0])


def match_probs(team1, team2):
    """Sieg / Unentschieden / Niederlage Wahrscheinlichkeiten."""
    g1 = predict_goals(team1, team2)
    g2 = predict_goals(team2, team1)

    matrix = np.outer(
        poisson.pmf(range(9), g1),
        poisson.pmf(range(9), g2)
    )

    win1 = np.sum(np.tril(matrix, -1))
    draw = np.sum(np.diag(matrix))
    win2 = np.sum(np.triu(matrix,  1))

    print(f"{team1} Sieg:   {win1:.1%}")
    print(f"Unentschieden:  {draw:.1%}")
    print(f"{team2} Sieg:   {win2:.1%}")
    print(f"\nErwartete Tore: {team1} {g1:.2f} – {g2:.2f} {team2}")


def top_results(team1, team2, max_goals=8):
    """Die 10 wahrscheinlichsten Ergebnisse."""
    g1 = predict_goals(team1, team2)
    g2 = predict_goals(team2, team1)

    matrix = np.outer(
        poisson.pmf(range(max_goals + 1), g1),
        poisson.pmf(range(max_goals + 1), g2)
    )

    rows = [
        (i, j, matrix[i, j])
        for i in range(max_goals + 1)
        for j in range(max_goals + 1)
    ]
    rows.sort(key=lambda x: x[2], reverse=True)

    result = pd.DataFrame(rows[:10], columns=[team1, team2, 'Wahrscheinlichkeit'])
    result['Wahrscheinlichkeit'] = result['Wahrscheinlichkeit'].map('{:.2%}'.format)
    return result

## 7 · Beispiele

In [26]:
top_results('France', 'Ecuador')

,France,Ecuador,Wahrscheinlichkeit
0,1,0,18.58%
1,0,0,16.24%
2,1,1,12.51%
3,0,1,10.93%
4,2,0,10.63%
5,2,1,7.16%
6,1,2,4.21%
7,3,0,4.06%
8,0,2,3.68%
9,3,1,2.73%


In [25]:
match_probs('France', 'Ecuador')

France Sieg:   47.0%
Unentschieden:  31.4%
Ecuador Sieg:   21.6%

Erwartete Tore: France 1.14 – 0.67 Ecuador


## 8 · Nächste Ausbaustufe (Version 3)

- Elo mit Torabstand-Bonus
- Dixon-Coles Korrektur (Korrelation bei 0:0 / 1:0 / 0:1 / 1:1)
- Form der letzten 5 Spiele als Feature
- Automatische WM-Teilnehmerliste 2026
- Wettquoten-Features zur Kalibrierung